# Track 2 — Training from Scratch Orchestrator

Thin orchestrator only. All real logic lives in `track2_scratch/scripts/*.py`
(agent-editable, ordinary `.py` modules). Cells below just **sync code**, **install
deps**, and **call into those scripts**.

**Before running the sync cell:** after any local agent edit you MUST commit and push
(`git add -A && git commit -m ... && git push`) so the remote kernel pulls your
latest code. Re-running the sync cell picks up new edits.

> **GPU tip:** If you already have a Colab session running for Track 1, reuse it
> via *Auto Connect* — saves GPU queue wait time. If starting fresh, connect a
> T4/A100 runtime before running any cells.

## Cell 1 — Sync: clone or pull latest code

In [ ]:
!git clone https://github.com/DevaNandanJS/Benchmarking-LLM-fine-tuning-vs-training-from-scratch-using-the-same-dataset.git llm_task 2>/dev/null || (cd llm_task && git pull)
%cd llm_task

## Cell 2 — Install dependencies

Key Track 2 dep: `tokenizers` (Hugging Face Rust-backed BPE trainer).  
`torch` is NOT in requirements.txt — Colab ships a CUDA-enabled version already.

In [ ]:
!pip install -q -r requirements.txt
# Freeze the exact versions to track2_scratch/logs/environment.txt
# (reproducibility lock — commit this file back to the repo)
import os
os.makedirs('track2_scratch/logs', exist_ok=True)
!pip freeze > track2_scratch/logs/environment.txt
print('Dependency snapshot written to track2_scratch/logs/environment.txt')

## Cell 3 — Hardware check

Confirms a GPU is attached and logs key info. Script exits non-zero if no GPU found —
do NOT proceed with training cells if this cell fails.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected — connect a Colab GPU runtime first'
props = torch.cuda.get_device_properties(0)
print('GPU:         ', torch.cuda.get_device_name(0))
print('VRAM (GB):   ', round(props.total_memory / 1e9, 2))
print('torch:       ', torch.__version__)
import tokenizers
print('tokenizers:  ', tokenizers.__version__)

---

## Phase 1 — Custom Tokenizer Training

**Goal:** sweep vocab sizes 256 / 1024 / 4096, compute fertility for each,
select the best size using a diminishing-returns criterion, save the final
tokenizer files, and write `configs/tokenizer_choice.md` with actual numbers.

**Outputs produced:**
- `track2_scratch/eval/vocab_sweep.csv`
- `track2_scratch/tokenizer/vocab.json` + `merges.txt`
- `track2_scratch/configs/tokenizer_choice.md`
- `track2_scratch/configs/run_phase1_tokenizer.json`
- `track2_scratch/tokenizer_candidates/vocab{256,1024,4096}/` (all candidates saved)

**After this cell:** commit + push all of the above so they are version-controlled.

In [ ]:
!python track2_scratch/scripts/train_tokenizer.py

### Inspect sweep results

In [ ]:
import pandas as pd
df = pd.read_csv('track2_scratch/eval/vocab_sweep.csv')
print(df.to_string(index=False))
print()
print(open('track2_scratch/configs/tokenizer_choice.md').read())

---

## Phase 2 -- Dataset Construction

**Prerequisite:** Phase 1 must have run on Colab and been committed + pushed.
Specifically, `track2_scratch/tokenizer/vocab.json` and `merges.txt` must exist
in the repo after your `git pull`. If they are missing, run Phase 1 first.

**Inputs:**
- `track2_scratch/tokenizer/vocab.json` + `merges.txt` (from Phase 1)
- `data/extracted/document_clean.txt` (from Track 1 Phase 1)

**Outputs produced:**
- `data/processed/track2_train.pt`
- `data/processed/track2_val.pt`
- `data/processed/track2_dataset_stats.json`
- `track2_scratch/configs/run_phase2_dataset.json`
- `track2_scratch/configs/split_strategy.md`

**After this cell:** commit + push all outputs so they are version-controlled.

In [ ]:
!python track2_scratch/scripts/build_dataset.py

### Inspect Phase 2 results

In [ ]:
import json as _json
import torch

stats = _json.load(open("data/processed/track2_dataset_stats.json"))
print("=== Track 2 Dataset Stats ===")
for k, v in stats.items():
    sv = str(v)
    print(f"  {k}: {sv[:80]}..." if len(sv) > 80 else f"  {k}: {sv}")

train = torch.load("data/processed/track2_train.pt", weights_only=True)
val   = torch.load("data/processed/track2_val.pt",   weights_only=True)
print("\ntrain input_ids shape:", tuple(train["input_ids"].shape))
print("val   input_ids shape:", tuple(val["input_ids"].shape))
print("chars/window:", stats["chars_per_window"], "raw chars per training example")

print("\n=== Split Strategy (excerpt) ===")
print(open("track2_scratch/configs/split_strategy.md").read()[:800])


---

## Phase 3 — Model Architecture Implementation

**Goal:** Build and validate a from-scratch decoder-only Transformer. Reads
`vocab_size` from the Phase 1 tokenizer and `block_size` from Phase 2 stats
automatically; falls back to `vocab_size=1024, block_size=256` if those files
are not yet present (safe for local smoke-testing).

**Prerequisite:** Phase 1 and Phase 2 must have run on Colab and been committed
+ pushed so `git pull` in Cell 1 picks up `track2_scratch/tokenizer/vocab.json`
and `data/processed/track2_dataset_stats.json`. If those are missing the script
falls back to defaults and prints a warning — the unit tests still run.

**Outputs (written on test success):**
- `track2_scratch/configs/run_phase3_model.json` — architecture config dump
- `track2_scratch/configs/trainable_params.json` — per-component param count

**After this cell:** Verify all 8 tests pass, commit + push the two JSON
artifacts, then proceed to Phase 4.

In [ ]:
# Cell 3b — Run unit tests + dump architecture artifacts
# Runs all 8 unit tests unconditionally; writes JSON artifacts only on success.
# If any test raises AssertionError, execution stops here and Cell 3c will
# FileNotFoundError — fix the failing test before proceeding.
!python track2_scratch/scripts/model.py

In [ ]:
# Cell 3c — Inspect architecture config and parameter breakdown
# These files are written by Cell 3b AFTER all unit tests pass.
# FileNotFoundError here means Cell 3b failed — check its output above.
import json as _json

cfg = _json.load(open('track2_scratch/configs/run_phase3_model.json'))
print('=== Phase 3 Model Config ===')
for k, v in cfg.items():
    print(f'  {k}: {v}')

params = _json.load(open('track2_scratch/configs/trainable_params.json'))
print('\n=== Parameter Count ===')
for k, v in params.items():
    if isinstance(v, int):
        print(f'  {k}: {v:,}')
    else:
        print(f'  {k}: {v}')

---

## Phase 4 — Training Loop

**Prerequisites:** Phases 1–3 must have run on Colab and all outputs committed + pushed.
Specifically, these files must exist after `git pull` in Cell 1:
- `track2_scratch/tokenizer/vocab.json` (Phase 1)
- `data/processed/track2_train.pt` and `track2_val.pt` (Phase 2)
- `track2_scratch/configs/trainable_params.json` (Phase 3)

**Sweep runs (one cell each):**
| Run | n_layer | n_embd | lr | Sweep axis |
|---|---|---|---|---|
| `small` | 4 | 128 | 3e-4 | Architecture |
| `base` | 6 | 192 | 3e-4 | Architecture |
| `base_highlr` | 6 | 192 | 6e-4 | Learning rate |

**Outputs produced (commit after all 3 runs):**
- `track2_scratch/configs/run_phase4_<run>.json` — config dump (pre-training)
- `track2_scratch/logs/<run>/metrics.jsonl` — step-level loss/LR log
- `track2_scratch/checkpoints/best_val/<run>/best_val.pt` — best checkpoint
- `track2_scratch/checkpoints/best_val/<run>/best_val_config.json`
- `track2_scratch/checkpoints/last/<run>/last_ckpt.pt` — final-step audit artifact
- `track2_scratch/eval/sweep_results.csv` — one row per run

> **After all 3 runs:** `git add -A && git commit -m 'Phase 4: training complete' && git push`

### Cell 4b — Smoke-test (local pre-flight, CPU, ~5 seconds)

Validates all code paths (shapes, loss, gradients, LR schedule, `evaluate()`).
Does **not** write checkpoints or update `sweep_results.csv`.

In [ ]:
# Smoke-test: 4 chunks, 5 steps, CPU fp32.
# Run this locally before pushing to Colab to catch shape/dtype bugs early.
!python track2_scratch/scripts/train.py --run base --smoke-test

### Cell 4c — Run `small` (architecture sweep: 4 layers, 128-embd, lr=3e-4)

**Expected:** initial loss ≈ ln(1024) ≈ 6.93 (random-model baseline), then descent.
Smaller architecture — expect faster overfitting than `base`.

In [ ]:
!python track2_scratch/scripts/train.py --run small

### Cell 4d — Run `base` (architecture sweep: 6 layers, 192-embd, lr=3e-4)

In [ ]:
!python track2_scratch/scripts/train.py --run base

### Cell 4e — Run `base_highlr` (LR sweep: same arch as `base`, lr=6e-4)

Same architecture as `base` — isolates the effect of a 2x higher learning rate.
Expect faster initial descent but potentially earlier/worse overfitting.

In [ ]:
!python track2_scratch/scripts/train.py --run base_highlr

### Cell 4f — Inspect sweep results

In [ ]:
import pandas as pd
df = pd.read_csv('track2_scratch/eval/sweep_results.csv')
print(df.to_string(index=False))
print(f"\nBest run: {df.loc[df['best_val_loss'].idxmin(), 'run_name']}  "
      f"(best_val_loss={df['best_val_loss'].min():.4f})")

### Cell 4g — Inspect best checkpoint configs

In [ ]:
import json as _json, os
for run in ['small', 'base', 'base_highlr']:
    cfg_path = f'track2_scratch/checkpoints/best_val/{run}/best_val_config.json'
    if os.path.exists(cfg_path):
        cfg = _json.load(open(cfg_path))
        print(f"\n=== {run} best checkpoint ===")
        for k in ['run_name', 'n_layer', 'n_embd', 'learning_rate',
                  'best_val_loss', 'best_val_step', 'total_steps', 'timestamp']:
            print(f"  {k}: {cfg.get(k, 'N/A')}")
    else:
        print(f"\n[{run}] best_val_config.json not found -- run Cell 4c/4d/4e first")

## Phase 5 — Quantitative Evaluation

*Cells to be added when Phase 5 scripts are ready.*

## Phase 6 — Qualitative Evaluation (Generation)

*Cells to be added when Phase 6 scripts are ready.*

## Phase 7 — Cross-Track Comparison

*Cells to be added when Phase 7 scripts are ready.*